# Train Eve on Kaggle

Pretrains Eve on a Kaggle GPU, then (optionally) fine-tunes it (the joint recipe), evaluates and
calibrates it.
Everything worth keeping (the tokenizer, the token files and the runs) is written to
`/kaggle/working`, so it is saved as this notebook's output.

## One-time setup

1. **Push your branch.** Kaggle clones the code from GitHub, so it trains what is pushed.
2. **Import this notebook:** on Kaggle, *Create → New Notebook → File → Import Notebook* and
   upload this file. Keep the notebook **private** (the default).
3. **Settings** (right sidebar):
   - *Accelerator*: **GPU T4 x2**. Eve trains on one GPU. Don't pick P100: current PyTorch
     builds no longer support it.
   - *Internet*: **on** (it needs a phone-verified Kaggle account).
4. **Secrets** (*Add-ons → Secrets*), then tick *Attach to notebook* for each:
   - `GITHUB_TOKEN`: a fine-grained GitHub token with read-only *Contents* access to
     `denskrlv/eve-lm` only.
   - `HF_TOKEN` (optional): a Hugging Face read token, after accepting the terms of
     [xlam-function-calling-60k](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k).
     Without it, fine-tuning data is built without xLAM.
5. **Your tokenizer (optional, recommended):** upload `artifacts/tokenizer/eve-8k.json` as a
   **private** dataset and add it as input. Then Kaggle's runs use the same tokens as your local
   ones; without it, the notebook trains a new tokenizer (slower, and different from your local one).

## Running

- Set `CONFIG` below, then *Save Version → Save & Run All (Commit)*. It runs in the background,
  even with the browser closed, for up to 12 hours; the results are under the version's *Output*.
- Training stops itself before the 12-hour limit. To continue, start a new version with
  *Add Input → Your Work → this notebook's output*: the notebook copies it back and resumes
  from `last.pt`.
- When it's done, download `artifacts/runs/...` from the *Output* tab into your local
  `artifacts/runs/`.
- **TinyTools:** with `TINYTOOLS` set to a glob of parquet files (a `text` column, e.g. the
  round-1 converters' output), the notebook mixes them into the train tokens at `MIX_SHARE`
  (`eve corpus encode --mix`). The files must be in the clone or restored from an input. Its token
  files, pretraining run and fine-tune get a `-tinytools` suffix, so they never mix with a web-only
  run. Leave `TINYTOOLS = ""` for web text only.

**Privacy:** the token is read from Kaggle Secrets and never printed or saved. Keep the
notebook, its output and any datasets you upload private.

In [ ]:
import time

START = time.time()  # Kaggle's 12-hour limit counts from about here

CONFIG = "small-250m"  # configs/experiments/<name>.toml: small-250m or base-900m
BRANCH = "main"  # the branch Kaggle clones: push your changes first
TINYTOOLS = ""  # a glob of parquet files to mix into the train text, e.g. "data/tinytools/*.parquet"
MIX_SHARE = 0.1  # their share of the train text: ~25M of 250M tokens
FINETUNE = True  # after pretraining finishes: build the task data, fine-tune and evaluate
SESSION_HOURS = 11.5  # stop training before the limit, so the outputs still get saved

REPO_URL = "github.com/denskrlv/eve-lm.git"
REPO = "/tmp/eve-lm"  # code, venv and downloads: thrown away after the session
WORK = "/kaggle/working"  # artifacts and token files: saved as the notebook's output

In [ ]:
import base64
import json
import os
import shutil
import subprocess
import sys
import tomllib
from pathlib import Path


def sh(*cmd, cwd=REPO, ok=(0,), show=None):
    """Run a command and stream its output; fail if it exits with a code not in ``ok``.
    ``show`` is what to print instead of the command, to keep secrets out of the log."""
    cmd = [str(part) for part in cmd]
    print("$", show or " ".join(cmd), flush=True)
    env = {**os.environ, "COLUMNS": "120", "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() not in ok:
        raise RuntimeError(f"{show or cmd[0]} exited with code {proc.returncode}")
    return proc.returncode


def eve(*args, budget=False):
    """Run the ``eve`` command line. With ``budget``, stop it when the session runs out of time
    (the runs save every 500 steps, so they continue from there with --resume)."""
    show = " ".join(["eve", *map(str, args)])
    cmd = [f"{REPO}/.venv/bin/eve", *args]
    if not budget:
        return sh(*cmd, show=show)
    seconds = int(SESSION_HOURS * 3600 - (time.time() - START))
    if seconds < 600:
        raise RuntimeError("less than 10 minutes left in this session: run a new version")
    print(f"time budget: {seconds / 3600:.1f} h")
    return sh("timeout", seconds, *cmd, ok=(0, 124), show=show)  # 124: out of time


def secret(name):
    """A Kaggle secret attached to this notebook, or None."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret(name)
    except Exception:
        return None


def sets(**sections):
    """``eve run``'s overrides: sets(run={"out": "x"}) -> ["--set", 'run.out="x"']."""
    return [
        arg
        for section, values in sections.items()
        for key, value in values.items()
        for arg in ("--set", f"{section}.{key}={json.dumps(value)}")
    ]


def count(text):
    """'250M' -> 250_000_000, as the eve command line reads it."""
    scale = {"k": 10**3, "m": 10**6, "b": 10**9}.get(text[-1].lower(), 1)
    return int(float(text[:-1] if scale > 1 else text) * scale)

## Code and environment

In [ ]:
token = secret("GITHUB_TOKEN")
if not token:
    raise RuntimeError("add a GITHUB_TOKEN secret (Add-ons → Secrets) and attach it")

# The token goes in a header for this one command, so it never lands in .git/config or the log.
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
shutil.rmtree(REPO, ignore_errors=True)
sh(
    "git",
    "-c",
    f"http.extraHeader=Authorization: Basic {auth}",
    "clone",
    "--depth",
    "1",
    "--branch",
    BRANCH,
    f"https://{REPO_URL}",
    REPO,
    cwd="/tmp",
    show=f"git clone --depth 1 --branch {BRANCH} https://{REPO_URL} {REPO}",
)
del token, auth
sh("git", "log", "-1", "--format=%h %s")

In [ ]:
# uv installs the exact versions in uv.lock: torch with CUDA, and Python 3.12 if needed.
sh(sys.executable, "-m", "pip", "install", "-q", "uv", cwd="/tmp")
sh(sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--extra", "train", show="uv sync --frozen --no-dev --extra train")

GPU_CHECK = """
import torch
assert torch.cuda.is_available(), "no GPU: Settings → Accelerator → GPU T4 x2"
name, (major, minor) = torch.cuda.get_device_name(), torch.cuda.get_device_capability()
print(f"torch {torch.__version__} · CUDA {torch.version.cuda} · {name} (sm_{major}{minor})")
assert major >= 7, f"this torch build doesn't support the {name}: choose GPU T4 x2"
"""
sh(f"{REPO}/.venv/bin/python", "-c", GPU_CHECK, show="python -c GPU_CHECK")
eve("info")

## Outputs and earlier sessions

In [ ]:
EXPERIMENT = f"configs/experiments/{CONFIG}.toml"
cfg = tomllib.loads((Path(REPO) / EXPERIMENT).read_text())
SUFFIX = "-tinytools" if TINYTOOLS else ""  # keeps a mixed run apart from a web-only one
DATA = cfg["data"]["path"] + SUFFIX  # the token files to train on, e.g. data/bin
RUN = f"artifacts/runs/{CONFIG}{SUFFIX}"  # where pretraining writes
TOKENIZER = "artifacts/tokenizer/eve-8k.json"

# artifacts/ and the token files live in /kaggle/working, so they are saved with the output.
for rel in ("artifacts", DATA):
    (Path(WORK) / rel).mkdir(parents=True, exist_ok=True)
    link = Path(REPO) / rel
    if not link.is_symlink():
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(Path(WORK) / rel, target_is_directory=True)


def copy_new(src, dst):
    """Copy a file unless this session already has it."""
    if not os.path.exists(dst):
        shutil.copy2(src, dst)


# An earlier session's output added as input: copy it back, so the runs resume.
for rel in ("artifacts", DATA):
    for src in sorted(Path("/kaggle/input").glob(f"**/{rel}")):
        if src.is_dir():
            print(f"restoring {src}")
            shutil.copytree(src, Path(WORK) / rel, dirs_exist_ok=True, copy_function=copy_new)

# A tokenizer uploaded as a dataset: use it rather than training a new one.
tokenizer = Path(REPO) / TOKENIZER
if not tokenizer.exists() and (found := sorted(Path("/kaggle/input").glob("**/eve-8k.json"))):
    print(f"using the tokenizer {found[0]}")
    tokenizer.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(found[0], tokenizer)

sh("ls", "-R", WORK, cwd=WORK)

## Pretraining data

In [ ]:
# The corpus each recipe's steps were counted on (base-900m reads 850M tokens of its 900M).
CORPUS_TOKENS = {"small-250m": "250M", "base-900m": "900M"}[CONFIG]
# A FineWeb-Edu shard is 2.15 GB, roughly 700M GPT-2 tokens; `eve corpus encode` warns if it runs out.
SHARDS = 1 if count(CORPUS_TOKENS) <= 400e6 else 2
FINEWEB = "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT"

have_tokenizer = (Path(REPO) / TOKENIZER).exists()
have_corpus = (Path(REPO) / DATA / "meta.json").exists()  # written last by `eve corpus encode`
if have_tokenizer and have_corpus:
    print("the tokenizer and the token files are already here")
else:
    raw = Path(REPO) / "data/raw/fineweb-edu"
    raw.mkdir(parents=True, exist_ok=True)
    for i in range(SHARDS):
        name = f"{i:03d}_00000.parquet"
        sh("curl", "-fsSL", "--retry", "3", "-o", raw / name, f"{FINEWEB}/{name}")
    if not have_tokenizer:
        eve("tokenizer", "train")
    mix = []
    if TINYTOOLS:
        mix = ["--mix", TINYTOOLS, "--mix-share", MIX_SHARE]
    eve("corpus", "encode", "--out", DATA, "--train-tokens", CORPUS_TOKENS, "--workers", os.cpu_count(), *mix)
    shutil.rmtree(raw)  # the token files are all pretraining needs

## Pretraining

In [ ]:
if (Path(REPO) / RUN / "model.pt").exists():
    print(f"{RUN} is already done")
elif eve("run", EXPERIMENT, *sets(data={"path": DATA}, run={"out": RUN, "resume": True}), budget=True) == 124:
    print(
        "\nOut of session time. To continue, run a new version with this version's output added"
        " as input (Add Input → Your Work): it resumes from the last save."
    )

## Fine-tuning and evaluation

In [ ]:
FT_RUN = {"small-250m": "artifacts/runs/joint-small", "base-900m": "artifacts/runs/joint-base"}[CONFIG]
FT_RUN += SUFFIX
MASSIVE = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
XLAM = (
    "https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k"
    "/resolve/main/xlam_function_calling_60k.json"
)

ready = FINETUNE and (Path(REPO) / RUN / "model.pt").exists()
if not ready:
    print("skipping: FINETUNE is off, or pretraining isn't done")
else:
    raw = Path(REPO) / "data/raw"
    (raw / "massive").mkdir(parents=True, exist_ok=True)
    sh("bash", "-c", f"curl -fsSL --retry 3 {MASSIVE} | tar xz -C data/raw/massive")
    if hf_token := secret("HF_TOKEN"):
        (raw / "xlam").mkdir(parents=True, exist_ok=True)
        sh(
            "curl",
            "-fsSL",
            "--retry",
            "3",
            "-H",
            f"Authorization: Bearer {hf_token}",
            "-o",
            raw / "xlam/xlam_function_calling_60k.json",
            XLAM,
            show="curl xlam_function_calling_60k.json",
        )
    else:
        print("no HF_TOKEN secret: building the task data without xLAM")
    eve("data", "external")
    eve("data", "build")

    # The joint recipe, started from the run this session pretrained.
    overrides = sets(model={"init": f"{RUN}/model.pt"}, run={"out": FT_RUN, "resume": True})
    done = eve("run", "configs/experiments/joint-base.toml", *overrides, budget=True)
    if done == 124:
        print("\nOut of session time: run a new version with this output as input to continue.")
    else:
        names = [
            name
            for name in ("val", "heldout-templates", "massive-test")
            if (Path(REPO) / "data/task" / f"{name}.jsonl").exists()
        ]
        eve("evaluate", FT_RUN, "--sets", *names)
        eve("calibrate", FT_RUN, "--save")  # temperatures for familiar and for new tools

## What gets saved

In [ ]:
sh("du", "-ah", "--max-depth=4", WORK, cwd=WORK)